# Cosine Similarity Evaluation

## Import packages

In [70]:
import nltk
import os
import string
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass
from unidecode import unidecode
from rouge_score import rouge_scorer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_community.embeddings import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import OllamaEmbeddings
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

## Disable warnings

In [71]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

## Import packages

In [72]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Define evaluation function

In [73]:
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess(corpus: str) -> str:
  corpus = corpus.lower()
  stopset = nltk.corpus.stopwords.words('english') + nltk.corpus.stopwords.words('russian') + list(string.punctuation)
  tokens = nltk.word_tokenize(corpus)
  tokens = [t for t in tokens if t not in stopset]
  tokens = [lemmatizer.lemmatize(t) for t in tokens]
  corpus = ' '.join(tokens)
  corpus = unidecode(corpus)
  return corpus

In [74]:
embeddings = OllamaEmbeddings(model='llama3.1')
store = LocalFileStore("./.embeddings_cache")

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
  embeddings,
  store,
  namespace=embeddings.model,
)

In [75]:
def embeddings_cosine_sim_metric(expected_answers: list[str], predicted_answers: list[str]) -> float:
  results = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    expected_embedding = np.array(cached_embeddings.embed_query(expected_answer))
    predicted_embedding = np.array(cached_embeddings.embed_query(predicted_answer))

    sim = cosine_similarity(
      expected_embedding.reshape(1, -1),
      predicted_embedding.reshape(1, -1),
    )[0][0]

    results.append(sim)

  return np.mean(results)

In [76]:
smoothie_f = nltk.translate.bleu_score.SmoothingFunction().method4

def bleu_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    predicted_tokens = nltk.word_tokenize(predicted_answer)
    expected_tokens = [nltk.word_tokenize(expected_answer)]

    score = nltk.translate.bleu_score.sentence_bleu(
      expected_tokens,
      predicted_tokens,
      smoothing_function=smoothie_f,
    )

    scores.append(score)

  return np.mean(scores)

In [77]:
rogue_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rogue_l_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_l_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rougeL'])

  return np.mean(scores)

In [78]:
rogue_1_scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

def rogue_1_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_1_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rouge1'])

  return np.mean(scores)

In [79]:
def eval_rag(chain) -> float:
  dataset_df = pd.read_csv('../datasets/mediqa.csv')
  expected_answers = dataset_df['answer']
  predicted_answers = []

  for index, row in tqdm(list(dataset_df.iterrows()), desc='Questions'):
    question = row['question']
    llm_answer = chain.invoke({'query': question})
    predicted_answers.append(llm_answer)

  cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)

  return cos_score, bleu_score, rogue_1_score, rogue_l_score

## Define LLM

In [80]:
llm = ChatOpenAI(model='gpt-4o',temperature=0)

## Define chain

In [81]:
template = """
You are an assistant for question-answering tasks. Keep the answer verbose, with a minimum of three paragraphs.

QUERY: {query}

First, identify the key scientific concepts and data points that relate to the QUERY.
Then, analyze how these concepts connect to form a comprehensive answer.
Finally, synthesize your findings into a detailed response.
"""

prompt = PromptTemplate(
  template=template,
  input_variables=['query'],
)

chain = prompt | llm | StrOutputParser()

## Evaluate the model

Here we take only a subset of all MMLU subjects close to neurobiology.

In [82]:
questions = [
    'Describe NeuroActivityToolkit software',
]

for index, question in enumerate(questions):
    generation = chain.invoke(question)

    print(f'{index + 1}. {question}')
    print(generation)
    print('')

1. Describe NeuroActivityToolkit software
To address the query about the NeuroActivityToolkit software, we first need to identify the key scientific concepts and data points that are relevant. NeuroActivityToolkit is likely a software application designed for neuroscientific research, focusing on the analysis and visualization of neural activity data. Key concepts related to this software would include neural activity measurement techniques, data analysis methodologies, and visualization tools. Neural activity can be measured using various techniques such as electroencephalography (EEG), magnetoencephalography (MEG), functional magnetic resonance imaging (fMRI), or calcium imaging in animal models. The software would need to handle large datasets, perform complex statistical analyses, and provide intuitive visualizations to help researchers interpret the data.

The connection between these concepts is crucial for understanding the functionality and utility of the NeuroActivityToolkit. 

In [83]:
eval_rag(chain)

Questions:  47%|████▋     | 9/19 [01:12<01:20,  8.04s/it]


KeyboardInterrupt: 